In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
import torch

# 1. Load dataset
try:
    # sep='\t' for your tab-separated data
    df = pd.read_csv("data.csv", sep='\t', names=['label', 'text'], on_bad_lines='skip')
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except Exception as e:
    print(f"Error loading CSV: {e}")
    raise

# 2. Clean data and map labels
df = df.dropna(subset=['label', 'text'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
# Ensure we only keep rows that mapped correctly
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# 3. Convert to HuggingFace dataset
dataset = Dataset.from_pandas(df, preserve_index=False)

# 4. Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# 5. Tokenization
def tokenize(batch):
    # Tokenize the text
    result = tokenizer(
        [str(t) for t in batch['text']],
        truncation=True,
        padding='max_length',
        max_length=128
    )
    # Important: The Trainer looks for "labels" (plural)
    result['labels'] = batch['label']
    return result

print("Tokenizing dataset...")
# We remove 'text' and 'label' (singular) because we created 'labels' (plural)
# This prevents the Arrow length mismatch error
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names
)

# 6. Set format for PyTorch
tokenized_dataset.set_format("torch")

# 7. Train-test split
split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

# 8. Load model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# 9. Training arguments
training_args = TrainingArguments(
    output_dir="./model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

# 10. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset['test'],
)

# 11. Train
print("Starting training...")
trainer.train()

# 12. Save
trainer.save_model("./model")
tokenizer.save_pretrained("./model")
print("Success! Model saved to ./model")

Dataset loaded successfully. Shape: (5572, 2)
Tokenizing dataset...


Map:   0%|          | 0/5572 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
